In [3]:
import re
import requests
import warnings

import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)


JUDGE_MODEL = "mistral"
JUDGE_NAME  = "mistral"

CONTROL_FILE = "modfied_controlled_agents_llama3.csv"      # pre-debate preference responses
DEBATE_FILE  = "new_debate_results_llama3.csv"     # LLaMA3 debate + reflection
OUTPUT_FILE  = f"llama3_dialogue_{JUDGE_NAME}_scores.csv"    # output


# ================== Judge via Ollama (/api/chat) ==================
OLLAMA_CHAT_URL = "http://localhost:11434/api/chat"


def get_agreement_score(text_a: str, text_b: str, context: str = "preference"):

    prompt = f"""
You are an impartial evaluator.
Rate how much these two people agree with each other (1–5)
based on their {context} statements.

1 = totally disagree
5 = totally agree

A: {text_a}
B: {text_b}

Please output ONLY a single number (1–5).
Do not output any explanation.
"""

    try:
        r = requests.post(
            OLLAMA_CHAT_URL,
            json={
                "model": JUDGE_MODEL,
                "stream": False,
                "messages": [
                    {"role": "user", "content": prompt}
                ],
                "options": {
                    "temperature": 0.0
                },
            },
            timeout=120,
        )
        data = r.json()

        msg = data.get("message", {}) or {}
        text = (msg.get("content", "") or "").strip()

        if not text:
            print(f"⚠️ Empty response from {JUDGE_MODEL}:", data)
            return None

        matches = re.findall(r"[1-5]", text)
        if matches:
            return int(matches[-1])

        print(f"⚠️ No digit 1–5 in response from {JUDGE_MODEL}:", repr(text[:200]))
        return None

    except Exception as e:
        print(f"⚠️ Error calling judge model {JUDGE_MODEL}:", e)
        return None


# ================== Load controlled agents ==================
controlled = pd.read_csv(CONTROL_FILE, encoding="utf-8")
print(f"✅ Loaded {len(controlled)} controlled agents from '{CONTROL_FILE}'")


def _norm_str(x):
    """Normalize a possibly-NaN value to a lowercase string; NaN/None -> ''."""
    if x is None:
        return ""
    if isinstance(x, float) and pd.isna(x):
        return ""
    s = str(x).strip()
    if s.lower() in {"nan", "none"}:
        return ""
    return s.lower()


def clean_preference_response(raw: str) -> str:
    """
    Clean up noisy Preference_Response text:
      - remove everything after 'Preference score: X'
      - remove obvious technical/debug phrases
      - collapse weird punctuation runs like '.).).).)'
    """
    if raw is None:
        return ""

    text = str(raw).replace("\n", " ").strip()

    # Cut off at 'Preference score: <digit>' if present
    parts = re.split(r"(?i)Preference score\s*:\s*\d", text, maxsplit=1)
    if len(parts) > 1:
        text = parts[0]

    # Remove some known debug / meta phrases
    patterns_to_remove = [
        r"(?i)No code needed for this",
        r"(?i)It then prints out the stance using the dictionary",
        r"(?i)End of note\.",
        r"\[/edited\]",
    ]
    for pat in patterns_to_remove:
        text = re.sub(pat, " ", text)

    # Remove very long sequences of ).).). or similar
    text = re.sub(r"[.)]{5,}", " ", text)

    # Compress multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


def find_pref_response(agent: dict, topic_id: int):
    """
    Match this agent to a row in controlled_agents_llama3.csv.

    Strategy:
      1) Start with all rows having the same topic_id.
      2) If occupation is available in this topic, filter by occupation.
      3) If still >1 and region is available, further filter by region.
      4) If still >1 and pref is known, pick the row whose topic_preference
         is closest to agent['pref'].

    Returns (cleaned_Preference_Response, match_source), where match_source is
    'matched' or 'missing'.
    """
    occ = _norm_str(agent.get("occupation", ""))
    region = _norm_str(agent.get("region", ""))
    pref = agent.get("pref", None)

    base = controlled[controlled["topic_id"] == topic_id].copy()
    if base.empty:
        return None, "missing"

    candidates = base

    if occ and "occupation" in base.columns:
        mask_occ = base["occupation"].str.strip().str.lower() == occ
        if mask_occ.any():
            candidates = base[mask_occ]

    if len(candidates) > 1 and region and "region" in base.columns:
        mask_reg = candidates["region"].str.strip().str.lower() == region
        if mask_reg.any():
            candidates = candidates[mask_reg]

    if (
        len(candidates) > 1
        and pref is not None
        and "topic_preference" in candidates.columns
    ):
        candidates = candidates.copy()
        candidates["pref_diff"] = (
            candidates["topic_preference"] - int(pref)
        ).abs()
        candidates = candidates.sort_values("pref_diff").head(1)

    if len(candidates) > 0:
        raw_resp = candidates.iloc[0]["Preference_Response"]
        cleaned = clean_preference_response(raw_resp)
        return cleaned if cleaned else str(raw_resp), "matched"
    else:
        return None, "missing"


# ================== Load LLaMA3 debate results ==================
debates = pd.read_csv(DEBATE_FILE, encoding="utf-8")
print(f"✅ Loaded {len(debates)} LLaMA3 debate rows from '{DEBATE_FILE}'")

results = []

for _, row in tqdm(
    debates.iterrows(), total=len(debates), desc=f"🔍 Scoring debates ({JUDGE_NAME})"
):
    try:
        topic_id = int(row["topic_id"])

        # ---- Build agent metadata dicts from CSV columns ----
        A = {
            "pref": row.get("A_pref"),
            "region": row.get("A_region"),
            "occupation": row.get("A_occupation"),
        }
        B = {
            "pref": row.get("B_pref"),
            "region": row.get("B_region"),
            "occupation": row.get("B_occupation"),
        }

        A_pref = A.get("pref")
        B_pref = B.get("pref")

        # ---- Get pre-debate preference responses ----
        A_resp, A_src = find_pref_response(A, topic_id)
        B_resp, B_src = find_pref_response(B, topic_id)

        # ---- Post-debate texts: use reflection fields ----
        A_post = row.get("reflection_A", "") or ""
        B_post = row.get("reflection_B", "") or ""

        # ---- Ask judge to score agreement before and after ----
        if A_resp is None or B_resp is None:
            pre_score = None
        else:
            pre_score = get_agreement_score(
                A_resp, B_resp, context="pre-debate preference"
            )

        post_score = get_agreement_score(
            A_post, B_post, context="post-debate reflection"
        )

        delta = (
            post_score - pre_score
            if (pre_score is not None and post_score is not None)
            else None
        )

        results.append(
            {
                "topic_id": topic_id,
                "A_pref": A_pref,
                "B_pref": B_pref,
                "A_region": A.get("region"),
                "B_region": B.get("region"),
                "A_occupation": A.get("occupation"),
                "B_occupation": B.get("occupation"),
                "A_pref_response": A_resp,
                "B_pref_response": B_resp,
                "A_match_source": A_src,
                "B_match_source": B_src,
                "pre_agreement": pre_score,
                "post_agreement": post_score,
                "Δagreement": delta,
                "reflection_A": A_post,
                "reflection_B": B_post,
            }
        )

    except Exception as e:
        print(f"⚠️ Error in row with topic_id={row.get('topic_id')}: {e}")
        continue


out_df = pd.DataFrame(results)

cols = [
    "topic_id",
    "A_pref",
    "B_pref",
    "A_region",
    "B_region",
    "A_occupation",
    "B_occupation",
    "A_pref_response",
    "B_pref_response",
    "A_match_source",
    "B_match_source",
    "pre_agreement",
    "post_agreement",
    "Δagreement",
    "reflection_A",
    "reflection_B",
]
out_df = out_df[cols]

out_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"\n✅ Done! Saved to '{OUTPUT_FILE}' ({len(out_df)} rows)")
print(out_df.head(5))

✅ Loaded 180 controlled agents from 'modfied_controlled_agents_llama3.csv'
✅ Loaded 90 LLaMA3 debate rows from 'new_debate_results_llama3.csv'


🔍 Scoring debates (mistral): 100%|█████████████| 90/90 [07:04<00:00,  4.72s/it]


✅ Done! Saved to 'llama3_dialogue_mistral_scores.csv' (90 rows)
   topic_id  A_pref  B_pref     A_region B_region  A_occupation  \
0         1       1       5  New Zealand    Kenya       Teacher   
1         1       1       5      Nigeria    Kenya  Psychologist   
2         1       1       5      Nigeria   France  Data analyst   
3         1       1       5       Canada   France    Accountant   
4         1       1       5  New Zealand   Canada       Teacher   

       B_occupation                                    A_pref_response  \
0     Social worker  Reason 1: I strongly reject the notion that ta...   
1     Social worker  Reason 1: Taxes are a form of oppression, forc...   
2            Writer  Reason 1: Taxes are a significant burden on in...   
3  Graphic designer  I strongly reject the notion that taxes help t...   
4        Accountant  Reason 1: I strongly reject the notion that ta...   

                                     B_pref_response A_match_source  \
0  Reason 1: As 

In [1]:
import re
import requests
import warnings

import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)


JUDGE_MODEL = "mistral"
JUDGE_NAME  = "mistral"

CONTROL_FILE = "modfied_controlled_agents_qwen2.csv"      # pre-debate preference responses
DEBATE_FILE  = "new_debate_results_qwen.csv"     # LLaMA3 debate + reflection
OUTPUT_FILE  = f"qwen_dialogue_{JUDGE_NAME}_scores.csv"    # output


# ================== Judge via Ollama (/api/chat) ==================
OLLAMA_CHAT_URL = "http://localhost:11434/api/chat"


def get_agreement_score(text_a: str, text_b: str, context: str = "preference"):

    prompt = f"""
You are an impartial evaluator.
Rate how much these two people agree with each other (1–5)
based on their {context} statements.

1 = totally disagree
5 = totally agree

A: {text_a}
B: {text_b}

Please output ONLY a single number (1–5).
Do not output any explanation.
"""

    try:
        r = requests.post(
            OLLAMA_CHAT_URL,
            json={
                "model": JUDGE_MODEL,
                "stream": False,
                "messages": [
                    {"role": "user", "content": prompt}
                ],
                "options": {
                    "temperature": 0.0
                },
            },
            timeout=120,
        )
        data = r.json()

        msg = data.get("message", {}) or {}
        text = (msg.get("content", "") or "").strip()

        if not text:
            print(f"⚠️ Empty response from {JUDGE_MODEL}:", data)
            return None

        matches = re.findall(r"[1-5]", text)
        if matches:
            return int(matches[-1])

        print(f"⚠️ No digit 1–5 in response from {JUDGE_MODEL}:", repr(text[:200]))
        return None

    except Exception as e:
        print(f"⚠️ Error calling judge model {JUDGE_MODEL}:", e)
        return None


# ================== Load controlled agents ==================
controlled = pd.read_csv(CONTROL_FILE, encoding="utf-8")
print(f"✅ Loaded {len(controlled)} controlled agents from '{CONTROL_FILE}'")


def _norm_str(x):
    """Normalize a possibly-NaN value to a lowercase string; NaN/None -> ''."""
    if x is None:
        return ""
    if isinstance(x, float) and pd.isna(x):
        return ""
    s = str(x).strip()
    if s.lower() in {"nan", "none"}:
        return ""
    return s.lower()


def clean_preference_response(raw: str) -> str:
    """
    Clean up noisy Preference_Response text:
      - remove everything after 'Preference score: X'
      - remove obvious technical/debug phrases
      - collapse weird punctuation runs like '.).).).)'
    """
    if raw is None:
        return ""

    text = str(raw).replace("\n", " ").strip()

    # Cut off at 'Preference score: <digit>' if present
    parts = re.split(r"(?i)Preference score\s*:\s*\d", text, maxsplit=1)
    if len(parts) > 1:
        text = parts[0]

    # Remove some known debug / meta phrases
    patterns_to_remove = [
        r"(?i)No code needed for this",
        r"(?i)It then prints out the stance using the dictionary",
        r"(?i)End of note\.",
        r"\[/edited\]",
    ]
    for pat in patterns_to_remove:
        text = re.sub(pat, " ", text)

    # Remove very long sequences of ).).). or similar
    text = re.sub(r"[.)]{5,}", " ", text)

    # Compress multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


def find_pref_response(agent: dict, topic_id: int):
    """
    Match this agent to a row in controlled_agents_llama3.csv.

    Strategy:
      1) Start with all rows having the same topic_id.
      2) If occupation is available in this topic, filter by occupation.
      3) If still >1 and region is available, further filter by region.
      4) If still >1 and pref is known, pick the row whose topic_preference
         is closest to agent['pref'].

    Returns (cleaned_Preference_Response, match_source), where match_source is
    'matched' or 'missing'.
    """
    occ = _norm_str(agent.get("occupation", ""))
    region = _norm_str(agent.get("region", ""))
    pref = agent.get("pref", None)

    base = controlled[controlled["topic_id"] == topic_id].copy()
    if base.empty:
        return None, "missing"

    candidates = base

    if occ and "occupation" in base.columns:
        mask_occ = base["occupation"].str.strip().str.lower() == occ
        if mask_occ.any():
            candidates = base[mask_occ]

    if len(candidates) > 1 and region and "region" in base.columns:
        mask_reg = candidates["region"].str.strip().str.lower() == region
        if mask_reg.any():
            candidates = candidates[mask_reg]

    if (
        len(candidates) > 1
        and pref is not None
        and "topic_preference" in candidates.columns
    ):
        candidates = candidates.copy()
        candidates["pref_diff"] = (
            candidates["topic_preference"] - int(pref)
        ).abs()
        candidates = candidates.sort_values("pref_diff").head(1)

    if len(candidates) > 0:
        raw_resp = candidates.iloc[0]["Preference_Response"]
        cleaned = clean_preference_response(raw_resp)
        return cleaned if cleaned else str(raw_resp), "matched"
    else:
        return None, "missing"


# ================== Load LLaMA3 debate results ==================
debates = pd.read_csv(DEBATE_FILE, encoding="utf-8")
print(f"✅ Loaded {len(debates)} LLaMA3 debate rows from '{DEBATE_FILE}'")

results = []

for _, row in tqdm(
    debates.iterrows(), total=len(debates), desc=f"🔍 Scoring debates ({JUDGE_NAME})"
):
    try:
        topic_id = int(row["topic_id"])

        # ---- Build agent metadata dicts from CSV columns ----
        A = {
            "pref": row.get("A_pref"),
            "region": row.get("A_region"),
            "occupation": row.get("A_occupation"),
        }
        B = {
            "pref": row.get("B_pref"),
            "region": row.get("B_region"),
            "occupation": row.get("B_occupation"),
        }

        A_pref = A.get("pref")
        B_pref = B.get("pref")

        # ---- Get pre-debate preference responses ----
        A_resp, A_src = find_pref_response(A, topic_id)
        B_resp, B_src = find_pref_response(B, topic_id)

        # ---- Post-debate texts: use reflection fields ----
        A_post = row.get("reflection_A", "") or ""
        B_post = row.get("reflection_B", "") or ""

        # ---- Ask judge to score agreement before and after ----
        if A_resp is None or B_resp is None:
            pre_score = None
        else:
            pre_score = get_agreement_score(
                A_resp, B_resp, context="pre-debate preference"
            )

        post_score = get_agreement_score(
            A_post, B_post, context="post-debate reflection"
        )

        delta = (
            post_score - pre_score
            if (pre_score is not None and post_score is not None)
            else None
        )

        results.append(
            {
                "topic_id": topic_id,
                "A_pref": A_pref,
                "B_pref": B_pref,
                "A_region": A.get("region"),
                "B_region": B.get("region"),
                "A_occupation": A.get("occupation"),
                "B_occupation": B.get("occupation"),
                "A_pref_response": A_resp,
                "B_pref_response": B_resp,
                "A_match_source": A_src,
                "B_match_source": B_src,
                "pre_agreement": pre_score,
                "post_agreement": post_score,
                "Δagreement": delta,
                "reflection_A": A_post,
                "reflection_B": B_post,
            }
        )

    except Exception as e:
        print(f"⚠️ Error in row with topic_id={row.get('topic_id')}: {e}")
        continue


out_df = pd.DataFrame(results)

cols = [
    "topic_id",
    "A_pref",
    "B_pref",
    "A_region",
    "B_region",
    "A_occupation",
    "B_occupation",
    "A_pref_response",
    "B_pref_response",
    "A_match_source",
    "B_match_source",
    "pre_agreement",
    "post_agreement",
    "Δagreement",
    "reflection_A",
    "reflection_B",
]
out_df = out_df[cols]

out_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"\n✅ Done! Saved to '{OUTPUT_FILE}' ({len(out_df)} rows)")
print(out_df.head(5))

✅ Loaded 180 controlled agents from 'modfied_controlled_agents_qwen2.csv'
✅ Loaded 90 LLaMA3 debate rows from 'new_debate_results_qwen.csv'


🔍 Scoring debates (mistral): 100%|█████████████| 90/90 [06:50<00:00,  4.56s/it]


✅ Done! Saved to 'qwen_dialogue_mistral_scores.csv' (90 rows)
   topic_id  A_pref  B_pref      A_region      B_region      A_occupation  \
0         1       1       5     Singapore        France     Social worker   
1         1       1       5  South Africa        France  Graphic designer   
2         1       1       5        Canada  South Africa        Accountant   
3         1       1       5         Egypt       Nigeria            Lawyer   
4         1       1       5     Singapore       Nigeria     Social worker   

   B_occupation                                    A_pref_response  \
0        Writer  Reason 1: As a social worker in Singapore, I s...   
1        Writer  Reason 1: As a graphic designer in South Afric...   
2  Entrepreneur  Reason 1: As an accountant in Canada, I strong...   
3  Psychologist  Reason 1: Taxes imposed by the government ofte...   
4  Psychologist  Reason 1: As a social worker in Singapore, I s...   

                                     B_pref_response 